# TRAIN stage: fine-tuning, test evaluation and latency (Colab)

Fine-tunes pretrained YOLO variants with the shared augmentation policy (`configs/augmentation.yaml`),
evaluates them on the held-out **test** split and measures latency. Equivalent to `scripts/train_eval.py`,
but meant for a Colab GPU.

**Before running**
1. `Runtime > Change runtime type > T4 GPU`.
2. Put two zips in your Google Drive (names and folder are set in the config cell):
   - the repo code: `configs/`, `src/`, `pyproject.toml`
   - `data/processed/` generated by `scripts/process_data.py` (`images/`, `labels/`, `data.yaml`)
3. Run the cells in order. Training takes about 2 h per model on a T4.

Results are written to Drive (`OUT`), so they survive a Colab disconnection.

## 1. Config (the only cell you should edit)

In [10]:
DRIVE_DIR = "/content/drive/MyDrive/Colab Notebooks"   # Drive folder that holds the two zips
REPO_ZIP = "repozip.zip"        # contains configs/, src/, pyproject.toml
DATA_ZIP = "processed.zip"      # contains the contents of data/processed
OUT = "/content/drive/MyDrive/VPC2/runs"   # where weights, curves and CSVs are saved

POLICY = "ultralytics_defaults" # arm of configs/augmentation.yaml
MODELS = ["yolov8s.pt", "yolov8n.pt"]
EPOCHS, BATCH, SEED = 50, 16, 42

## 2. Setup: Drive, dependencies, data and code

In [11]:
import glob, os, sys
from google.colab import drive
drive.mount("/content/drive")

!pip install -q ultralytics==8.4.172

import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"

!rm -rf /content/repo /content/processed
!unzip -q -o "{DRIVE_DIR}/{REPO_ZIP}" -d /content/repo
!unzip -q -o "{DRIVE_DIR}/{DATA_ZIP}" -d /content/processed

# Folder that contains images/ and labels/ (the zip may add one extra level)
DATA_ROOT = os.path.dirname(glob.glob("/content/processed/**/images", recursive=True)[0])

# data.yaml with the paths of THIS machine (the one from process_data.py has local paths)
open(f"{DATA_ROOT}/data.yaml", "w").write(f"""path: {DATA_ROOT}
train: images/train
val: images/val
test: images/test
names:
  0: BIODEGRADABLE
  1: CARDBOARD
  2: GLASS
  3: METAL
  4: PAPER
  5: PLASTIC
""")

# Make the repo code (vpc2) importable
REPO_ROOT = os.path.dirname(glob.glob("/content/repo/**/configs", recursive=True)[0])
sys.path.append(os.path.join(REPO_ROOT, "src"))
os.chdir(REPO_ROOT)
os.makedirs(OUT, exist_ok=True)
print("DATA_ROOT:", DATA_ROOT)
print("REPO_ROOT:", REPO_ROOT)
!ls {DATA_ROOT} {DATA_ROOT}/images

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 39.6 MB/s eta 0:00:00
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
DATA_ROOT: /content/processed/processed
REPO_ROOT: /content/repo/repozip
/content/processed/processed:
data.yaml  images  labels

/content/processed/processed/images:
test  train  val


## 3. Train (one run per model; same data, seed, epochs and augmentation)

In [12]:
from pathlib import Path
from ultralytics import YOLO
from vpc2.data.augmentation import load_policy

policy = load_policy(POLICY)
for w in MODELS:
    name = f"{Path(w).stem}_{POLICY}"          # e.g. yolov8s_proposed
    _ = YOLO(w).train(data=f"{DATA_ROOT}/data.yaml", epochs=EPOCHS, imgsz=640,
                      batch=BATCH, seed=SEED, workers=2, project=OUT, name=name,
                      exist_ok=True, **policy.train_kwargs())

Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/processed/processed/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8s_ultralytics_defaults, nbs=64, nms=None, opse

## 4. Evaluate on test and measure latency

- Metrics are on the **test** split. The confusion matrices saved during training are on **validation**.
- Latency = end to end per image (read, preprocess, inference, postprocess), one image at a time,
  200 test images after a 20-image warm-up. Always report it together with the hardware (here: Colab T4).

In [13]:
import time
import numpy as np
import pandas as pd

imgs = sorted(glob.glob(f"{DATA_ROOT}/images/test/*.*"))[:200]
metricas, latencias, por_clase = [], [], {}

for w in MODELS:
    name = f"{Path(w).stem}_{POLICY}"
    model = YOLO(f"{OUT}/{name}/weights/best.pt")

    m = model.val(data=f"{DATA_ROOT}/data.yaml", split="test", imgsz=640, plots=False)
    p, r = m.box.mp, m.box.mr
    metricas.append({"modelo": name, "mAP50": round(m.box.map50, 4), "mAP50-95": round(m.box.map, 4),
                     "precision": round(p, 4), "recall": round(r, 4), "F1": round(2 * p * r / (p + r), 4)})
    por_clase[name] = dict(zip(m.names.values(), m.box.maps.round(4)))

    for f in imgs[:20]:                                   # warm-up
        model.predict(f, imgsz=640, verbose=False)
    t_ms = []
    for f in imgs:
        torch.cuda.synchronize(); t0 = time.perf_counter()
        model.predict(f, imgsz=640, verbose=False)
        torch.cuda.synchronize(); t_ms.append((time.perf_counter() - t0) * 1000)
    latencias.append({"modelo": name, "ms_medio": round(float(np.mean(t_ms)), 2),
                      "ms_mediana": round(float(np.median(t_ms)), 2),
                      "FPS": round(1000 / float(np.mean(t_ms)), 1)})

tabla, lat, clases = pd.DataFrame(metricas), pd.DataFrame(latencias), pd.DataFrame(por_clase)
tabla.to_csv(f"{OUT}/tabla_metricas_{POLICY}.csv", index=False)
lat.to_csv(f"{OUT}/latencia_t4_{POLICY}.csv", index=False)
clases.to_csv(f"{OUT}/map_por_clase_{POLICY}.csv")
display(tabla); display(lat); display(clases)

Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 11,127,906 parameters, 0 gradients, 28.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 8.9±5.5 MB/s, size: 16.7 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/processed/processed/labels/test... 1045 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1045/1045 1.3Kit/s 0.8s
val: New cache created: /content/processed/processed/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 66/66 4.0it/s 16.4s
                   all       1045       7452       0.73      0.579      0.645      0.455
         BIODEGRADABLE        228       4504      0.782      0.528      0.629      0.359
             CARDBOARD        146        490       0.62      0.486      0.536      0

,modelo,mAP50,mAP50-95,precision,recall,F1
0,yolov8s_ultralytics_defaults,0.6450,0.4552,0.7303,0.5791,0.6460
1,yolov8n_ultralytics_defaults,0.6225,0.4353,0.6981,0.5531,0.6172


,modelo,ms_medio,ms_mediana,FPS
0,yolov8s_ultralytics_defaults,19.29,19.40,51.8
1,yolov8n_ultralytics_defaults,10.40,9.87,96.2


,yolov8s_ultralytics_defaults,yolov8n_ultralytics_defaults
BIODEGRADABLE,0.3587,0.3445
CARDBOARD,0.4076,0.3943
GLASS,0.6092,0.5923
METAL,0.5015,0.4723
PAPER,0.4054,0.3748
PLASTIC,0.4489,0.4335


## Outputs (in `OUT`)
- `<model>_<policy>/weights/best.pt`: trained weights (use these for inference)
- `<model>_<policy>/args.yaml`, `results.csv`, `results.png`, `confusion_matrix*.png`
- `tabla_metricas_<policy>.csv`, `map_por_clase_<policy>.csv`, `latencia_t4_<policy>.csv`

Download the folder from Drive and place it under `reports/runs/` in the repo.